# 品牌/竞品洞察分析 (Brand & Competitive Insights)

**数据源**: 4170条京东相机评论（佳能、尼康、索尼、富士）

**分析框架**:
- Part 1: 时间趋势分析（月度评论量、声量份额、情感趋势）
- Part 2: 品牌定位分析（主题分布、差异化关键词、四维定位图）
- Part 4: 功能偏好分析（基于LDA 4主题的品牌×主题交叉分析）

## 0. 环境配置与数据加载

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
from itertools import combinations
import os, warnings
warnings.filterwarnings('ignore')

SAVE_DIR = 'figures'
os.makedirs(SAVE_DIR, exist_ok=True)

# 中文字体（Windows: SimHei, Mac: PingFang SC, Linux: Noto Sans CJK SC）
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

BRAND_COLORS = {'佳能':'#E63946','富士':'#FF8C00','尼康':'#2A9D8F','索尼':'#457B9D'}
BRAND_EN = {'佳能':'Canon','富士':'Fujifilm','尼康':'Nikon','索尼':'Sony'}
BRANDS = ['佳能','尼康','索尼','富士']
TOPICS = ['Appearance & Satisfaction','Design & Portability','Logistics & Service','Performance & Quality']
TOPIC_SHORT = {'Appearance & Satisfaction':'Appearance\n& Satisfaction','Design & Portability':'Design\n& Portability','Logistics & Service':'Logistics\n& Service','Performance & Quality':'Performance\n& Quality'}

In [ ]:
df_label = pd.read_csv('camera_labeled.csv')
df_sent = pd.read_excel('camera_reviews_sentiment_analysis.xlsx')
df = df_label.copy()
for col in ['predicted_sentiment','positive_score','neutral_score','negative_score','referral_score','community_score','discussion_score']:
    df[col] = df_sent[col].values

def parse_time(val):
    try: return pd.to_datetime(val)
    except:
        try: return pd.to_datetime(f"2026-{val}")
        except: return pd.NaT

df['时间'] = df['时间'].apply(parse_time)
df = df[(df['时间'] >= '2022-01-01') & (df['topic_label'].notna())].copy()
df['year_month'] = df['时间'].dt.to_period('M')
print(f"数据集: {df.shape}, 时间: {df['时间'].min()} ~ {df['时间'].max()}")
print(f"\n品牌分布:\n{df['品牌'].value_counts()}")

---\n## Part 1: 时间趋势分析

In [ ]:
monthly = df.groupby(['year_month','品牌']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(15,5))
for brand in BRANDS:
    ax.plot(range(len(monthly)), monthly[brand].values, marker='o', markersize=3, label=BRAND_EN[brand], color=BRAND_COLORS[brand], linewidth=1.5)
ax.set_xticks(range(0,len(monthly),3)); ax.set_xticklabels([str(monthly.index[i]) for i in range(0,len(monthly),3)], rotation=45, fontsize=8)
ax.set_xlabel('Month'); ax.set_ylabel('Number of Reviews'); ax.set_title('Monthly Review Volume by Brand', fontsize=14, fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/p1_monthly_volume.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
monthly_pct = monthly.div(monthly.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(15,5))
x = range(len(monthly_pct)); bottom = np.zeros(len(monthly_pct))
for brand in BRANDS:
    vals = monthly_pct[brand].values; ax.fill_between(x, bottom, bottom+vals, label=BRAND_EN[brand], color=BRAND_COLORS[brand], alpha=0.75); bottom += vals
ax.set_xticks(range(0,len(monthly_pct),3)); ax.set_xticklabels([str(monthly_pct.index[i]) for i in range(0,len(monthly_pct),3)], rotation=45, fontsize=8)
ax.set_xlabel('Month'); ax.set_ylabel('Share of Voice (%)'); ax.set_title('Monthly Share of Voice by Brand', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=9); ax.set_ylim(0,105); plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/p1_share_of_voice.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
monthly_sent = df.groupby(['year_month','品牌'])['positive_score'].mean().unstack()
fig, ax = plt.subplots(figsize=(15,5))
for brand in BRANDS:
    vals = monthly_sent[brand].dropna(); smoothed = vals.rolling(3, min_periods=1).mean()
    ax.plot(range(len(smoothed)), smoothed.values, marker='.', markersize=3, label=BRAND_EN[brand], color=BRAND_COLORS[brand], linewidth=1.8)
ax.set_xticks(range(0,len(monthly_sent),3)); ax.set_xticklabels([str(monthly_sent.index[i]) for i in range(0,len(monthly_sent),3)], rotation=45, fontsize=8)
ax.set_xlabel('Month'); ax.set_ylabel('Avg. Positive Score (3-Month MA)'); ax.set_title('Monthly Positive Sentiment Trend (3-Month Moving Average)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/p1_sentiment_trend.png', dpi=150, bbox_inches='tight'); plt.show()

---\n## Part 2: 品牌定位分析

In [ ]:
topic_crosstab = pd.crosstab(df['品牌'], df['topic_label'])
topic_pct = topic_crosstab.div(topic_crosstab.sum(axis=1), axis=0) * 100

N = len(TOPICS); angles = [n/float(N)*2*np.pi for n in range(N)] + [0]
fig, ax = plt.subplots(figsize=(8,8), subplot_kw=dict(polar=True))
for brand in BRANDS:
    values = [topic_pct.loc[brand, t] for t in TOPICS] + [topic_pct.loc[brand, TOPICS[0]]]
    ax.plot(angles, values, 'o-', linewidth=2, label=BRAND_EN[brand], color=BRAND_COLORS[brand]); ax.fill(angles, values, alpha=0.1, color=BRAND_COLORS[brand])
ax.set_xticks(angles[:-1]); ax.set_xticklabels(TOPICS, fontsize=9)
ax.set_title('Topic Distribution by Brand (Normalized %)', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35,1.1), fontsize=10); plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/p2_topic_radar.png', dpi=150, bbox_inches='tight'); plt.show()
print('\n各品牌主题分布(%):\n', topic_pct.round(1))

### 差异化关键词分析（自动化停用词）\n\n**方法**: 统计每个品牌Top50高频词 → 取4品牌交集作为停用词 → 计算TF-IDF ratio找各品牌独有词

In [ ]:
brand_top_words = {}
for brand in BRANDS:
    texts = df[df['品牌']==brand]['segmented_text'].dropna(); all_words = ' '.join(texts).split()
    word_counts = Counter(all_words); top50 = [(w,c) for w,c in word_counts.most_common(80) if len(w)>1][:50]
    brand_top_words[brand] = set([w for w,c in top50])
auto_stopwords = brand_top_words['佳能'] & brand_top_words['尼康'] & brand_top_words['索尼'] & brand_top_words['富士']
print(f'自动停用词({len(auto_stopwords)}个): {sorted(auto_stopwords)}')

brand_keywords = {}
for brand in BRANDS:
    bt = df[df['品牌']==brand]['segmented_text'].dropna().tolist(); ot = df[df['品牌']!=brand]['segmented_text'].dropna().tolist()
    tfidf = TfidfVectorizer(max_features=3000, token_pattern=r'(?u)\\b\\w+\\b'); X = tfidf.fit_transform(bt+ot)
    bm = X[:len(bt)].mean(axis=0).A1; om = X[len(bt):].mean(axis=0).A1; diff = bm/(om+1e-6); features = tfidf.get_feature_names_out()
    valid = [i for i,w in enumerate(features) if len(w)>1 and w not in auto_stopwords and bm[i]>0.005]
    sorted_valid = sorted(valid, key=lambda i: diff[i], reverse=True); brand_keywords[brand] = [(features[i], diff[i]) for i in sorted_valid[:8]]
    print(f'{BRAND_EN[brand]}: {[w for w,_ in brand_keywords[brand]]}')

all_kw = set()
for kws in brand_keywords.values(): all_kw.update([w for w,_ in kws])
kw_list = sorted(all_kw)
kw_matrix = pd.DataFrame(0.0, index=[BRAND_EN[b] for b in BRANDS], columns=kw_list)
for brand in BRANDS:
    texts = df[df['品牌']==brand]['segmented_text'].dropna()
    for kw in kw_list: kw_matrix.loc[BRAND_EN[brand], kw] = texts.str.contains(kw, na=False).sum()/len(texts)*100

fig, ax = plt.subplots(figsize=(16,5))
sns.heatmap(kw_matrix, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, linewidths=0.5, cbar_kws={'label':'Mention Rate (%)'})
ax.set_title('Brand-Specific Differential Keywords (Auto-filtered Stopwords)', fontsize=14, fontweight='bold'); ax.set_ylabel('Brand')
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/p2_differential_keywords.png', dpi=150, bbox_inches='tight'); plt.show()

### 品牌定位图（四维）

In [ ]:
topic_norm = topic_pct.copy()
for col in TOPICS: topic_norm[col] = (topic_pct[col]-topic_pct[col].min())/(topic_pct[col].max()-topic_pct[col].min()+1e-9)
fig, ax = plt.subplots(figsize=(12,6))
x_pos = range(len(TOPICS))
for brand in BRANDS:
    values = [topic_norm.loc[brand, t] for t in TOPICS]
    ax.plot(x_pos, values, 'o-', linewidth=2.5, markersize=10, label=BRAND_EN[brand], color=BRAND_COLORS[brand])
    for i, (xp,v) in enumerate(zip(x_pos, values)):
        ax.annotate(f'{topic_pct.loc[brand, TOPICS[i]]:.1f}%', (xp,v), textcoords='offset points', xytext=(8,5), fontsize=8, color=BRAND_COLORS[brand], fontweight='bold')
ax.set_xticks(x_pos); ax.set_xticklabels([TOPIC_SHORT[t] for t in TOPICS], fontsize=10)
ax.set_ylabel('Normalized Score (0-1)'); ax.set_title('Brand Positioning — Parallel Coordinates (All 4 Topics)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper right'); ax.grid(axis='y', alpha=0.3); ax.set_ylim(-0.1,1.15)
for xp in x_pos: ax.axvline(x=xp, color='gray', alpha=0.3)
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/p2_parallel_coordinates.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
topic_pairs = list(combinations(range(len(TOPICS)), 2)); review_count = df.groupby('品牌').size()
short_map = {'Appearance & Satisfaction':'Appearance','Design & Portability':'Design & Port.','Logistics & Service':'Logistics','Performance & Quality':'Performance'}
fig, axes = plt.subplots(2, 3, figsize=(18,11))
for idx, (i,j) in enumerate(topic_pairs):
    ax = axes.flatten()[idx]
    for brand in BRANDS:
        ax.scatter(topic_pct.loc[brand,TOPICS[i]], topic_pct.loc[brand,TOPICS[j]], s=review_count[brand]*0.35, color=BRAND_COLORS[brand], alpha=0.7, edgecolors='black', linewidth=0.8)
        ax.annotate(BRAND_EN[brand], (topic_pct.loc[brand,TOPICS[i]], topic_pct.loc[brand,TOPICS[j]]), textcoords='offset points', xytext=(8,8), fontsize=9, fontweight='bold', color=BRAND_COLORS[brand])
    ax.axhline(y=topic_pct[TOPICS[j]].mean(), color='gray', linestyle='--', alpha=0.4)
    ax.axvline(x=topic_pct[TOPICS[i]].mean(), color='gray', linestyle='--', alpha=0.4)
    ax.set_xlabel(f'{short_map[TOPICS[i]]} (%)', fontsize=9); ax.set_ylabel(f'{short_map[TOPICS[j]]} (%)', fontsize=9); ax.grid(alpha=0.3)
fig.suptitle('Brand Positioning — Pairwise Scatter Matrix (All 6 Combinations)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/p2_scatter_matrix.png', dpi=150, bbox_inches='tight'); plt.show()

---\n## Part 4: 功能偏好分析 (基于LDA 4主题)\n\n直接使用LDA主题分类模型的输出作为功能维度，无需额外构建关键词字典。\n4个主题维度完全由数据驱动，每条评论的主题标签由LDA模型+分类器自动分配。

In [ ]:
# 构建 品牌×主题 的情感得分和分布矩阵
hm_sent = pd.DataFrame(index=[BRAND_EN[b] for b in BRANDS], columns=TOPICS, dtype=float)
hm_pct = pd.DataFrame(index=[BRAND_EN[b] for b in BRANDS], columns=TOPICS, dtype=float)
for brand in BRANDS:
    bdf = df[df['品牌']==brand]
    for topic in TOPICS:
        mask = bdf['topic_label']==topic
        hm_sent.loc[BRAND_EN[brand], topic] = bdf.loc[mask, 'positive_score'].mean() if mask.sum()>0 else np.nan
        hm_pct.loc[BRAND_EN[brand], topic] = mask.sum()/len(bdf)*100
print('品牌×主题 情感得分:'); print(hm_sent.astype(float).round(3))
print('\n品牌×主题 分布(%):'); print(hm_pct.astype(float).round(1))

In [ ]:
# 品牌×主题 情感得分热力图
fig, ax = plt.subplots(figsize=(12,5))
sns.heatmap(hm_sent.astype(float), annot=True, fmt='.3f', cmap='RdYlGn', ax=ax, linewidths=0.5, vmin=0.88, vmax=0.96, cbar_kws={'label':'Avg. Positive Score'})
ax.set_title('LDA Topic × Brand: Average Positive Sentiment Score', fontsize=14, fontweight='bold')
ax.set_ylabel('Brand'); ax.set_xlabel('LDA Topic'); plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/p4_topic_sentiment_heatmap.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# 品牌×主题 分布热力图
fig, ax = plt.subplots(figsize=(12,5))
sns.heatmap(hm_pct.astype(float), annot=True, fmt='.1f', cmap='Blues', ax=ax, linewidths=0.5, cbar_kws={'label':'Topic Share (%)'})
ax.set_title('LDA Topic × Brand: Topic Distribution (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Brand'); ax.set_xlabel('LDA Topic'); plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/p4_topic_distribution_heatmap.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# 主题偏好品牌对比
fig, ax = plt.subplots(figsize=(12,6))
x = np.arange(len(TOPICS)); width = 0.2
for i, brand in enumerate(BRANDS):
    vals = [hm_pct.loc[BRAND_EN[brand], t] for t in TOPICS]
    ax.bar(x+i*width, vals, width, label=BRAND_EN[brand], color=BRAND_COLORS[brand], alpha=0.85)
ax.set_xticks(x+width*1.5); ax.set_xticklabels([TOPIC_SHORT[t] for t in TOPICS], fontsize=9)
ax.set_ylabel('Topic Share (%)'); ax.set_title('Topic Preference Comparison by Brand', fontsize=14, fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/p4_topic_by_brand.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# 主题×品牌 情感得分分组柱状图
fig, ax = plt.subplots(figsize=(12,6))
for i, brand in enumerate(BRANDS):
    vals = [hm_sent.loc[BRAND_EN[brand], t] for t in TOPICS]
    ax.bar(x+i*width, vals, width, label=BRAND_EN[brand], color=BRAND_COLORS[brand], alpha=0.85)
ax.set_xticks(x+width*1.5); ax.set_xticklabels([TOPIC_SHORT[t] for t in TOPICS], fontsize=9)
ax.set_ylabel('Avg. Positive Score'); ax.set_title('Sentiment Score by Brand × Topic', fontsize=14, fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3); ax.set_ylim(0.88, 0.96); plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/p4_sentiment_by_brand_topic.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# 主题关注趋势
topic_dummies = pd.get_dummies(df['topic_label'])
for t in TOPICS: df[f'topic_{t}'] = topic_dummies[t] if t in topic_dummies.columns else 0
monthly_topic = df.groupby('year_month')[[f'topic_{t}' for t in TOPICS]].mean()*100
TOPIC_COLORS = ['#E63946','#457B9D','#2A9D8F','#FF8C00']

fig, ax = plt.subplots(figsize=(15,5))
for i, topic in enumerate(TOPICS):
    vals = pd.Series(monthly_topic[f'topic_{topic}'].values).rolling(3, min_periods=1).mean()
    ax.plot(range(len(vals)), vals.values, marker='.', markersize=3, linewidth=1.8, label=topic, color=TOPIC_COLORS[i])
ax.set_xticks(range(0,len(monthly_topic),3))
ax.set_xticklabels([str(monthly_topic.index[i]) for i in range(0,len(monthly_topic),3)], rotation=45, fontsize=8)
ax.set_xlabel('Month'); ax.set_ylabel('Topic Share (%, 3-Month MA)')
ax.set_title('Topic Attention Trend Over Time (3-Month Moving Average)', fontsize=14, fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/p4_topic_trend.png', dpi=150, bbox_inches='tight'); plt.show()

---\n## 品牌优劣势总结

In [ ]:
for brand in BRANDS:
    bdf = df[df['品牌']==brand]
    pct_row = hm_pct.loc[BRAND_EN[brand]].astype(float)
    sent_row = hm_sent.loc[BRAND_EN[brand]].astype(float)
    top_topic = pct_row.idxmax(); top_sent = sent_row.idxmax(); low_sent = sent_row.idxmin()
    print(f"\n{'='*50}")
    print(f"【{BRAND_EN[brand]}】 Reviews: {len(bdf)}")
    print(f"  Avg Positive: {bdf['positive_score'].mean():.4f} | Avg Referral: {bdf['referral_score'].mean():.4f}")
    print(f"  Most discussed topic: {top_topic} ({pct_row[top_topic]:.1f}%)")
    print(f"  Highest sentiment topic: {top_sent} ({sent_row[top_sent]:.3f})")
    print(f"  Lowest sentiment topic: {low_sent} ({sent_row[low_sent]:.3f})")